In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
from tqdm import tqdm

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

from roi_classifier.prepare_data import prepare_roi_data
from roi_classifier.annotate_data import annotate_rois
from roi_classifier.train_classifier import train_roi_classifier




In [2]:
DATASET_ROOT = Path(r"E:\GCaMP6s_EX357")  # TODO: set this to your data path
assert DATASET_ROOT.exists(), f"Dataset root {DATASET_ROOT} does not exist."

ROI_DIR = PROJECT_ROOT / "data"
ROI_DIR.mkdir(parents=True, exist_ok=True)

ROI_DATA_PATH = ROI_DIR / "ex357_roi_features.npy"
ROI_DATA_PATH_2 = ROI_DIR / "all_roi_features.npy"

MODEL_OUT_DIR = PROJECT_ROOT / "models"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = PROJECT_ROOT / "config/classifier_config.yaml"

print(f"Extracting fluorescence data from {DATASET_ROOT.__str__()}")
print(f"Saving engineered data to {ROI_DATA_PATH.__str__()}")
print(f"Saving models to {MODEL_OUT_DIR.__str__()}")
print(f"Configuring classifier according to {CONFIG_PATH.__str__()}")

Extracting fluorescence data from E:\GCaMP6s_EX357
Saving engineered data to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\ex357_roi_features.npy
Saving models to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models
Configuring classifier according to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\classifier_config.yaml


In [7]:
update = False # Change to false 
backup = False # Change as you wish; controls whether or not a backup of the original engineered data is saved


roi_data = prepare_roi_data(
    dataset_root=DATASET_ROOT,
    input_file=ROI_DATA_PATH,
    output_file=ROI_DATA_PATH,
    update=update,
    backup=backup
)


Found 40 video directories under E:\GCaMP6s_EX357
  Processed 1-1-1: 372 ROIs
  Processed 1-1-2_0001: 469 ROIs
  Processed 1-1-3: 480 ROIs
  Processed 1-2-1: 378 ROIs
  Processed 1-2-2: 603 ROIs
  Processed 1-2-3: 512 ROIs
  Processed 1-3-1: 515 ROIs
  Processed 1-3-2: 319 ROIs
  Processed 1-3-3: 394 ROIs
  Processed 1-4-1: 460 ROIs
  Processed 1-4-2: 381 ROIs
  Processed 1-4-3: 428 ROIs
  Processed 2-1-1: 586 ROIs
  Processed 2-1-2: 178 ROIs
  Processed 2-1-3: 171 ROIs
  Processed 2-2-1: 395 ROIs
  Processed 2-2-2: 162 ROIs
  Processed 2-2-3: 150 ROIs
  Processed 2-3-1: 282 ROIs
  Processed 2-3-2: 225 ROIs
  Processed 2-3-3: 366 ROIs
  Processed 2-4-1: 315 ROIs
  Processed 2-4-2: 189 ROIs
  Processed 2-4-3: 376 ROIs
  Processed 2-1: 721 ROIs
  Processed 2-1_25uM_8m: 637 ROIs
  Processed 2-2_25uM_2m: 408 ROIs
  Processed 2-3: 394 ROIs
  Processed 2-3_25uM_14m: 373 ROIs
  Processed 2-1: 348 ROIs
  Processed 2-1_50uM_2m: 362 ROIs
  Processed 2-2: 247 ROIs
  Processed 2-3: 305 ROIs
  Proc

In [5]:
# Change these flags to control which ROIs are shown for annotation and how many
unlabeled_only = True 
labeled_only = False 
n_samples = 1000

assert not (unlabeled_only and labeled_only), "unlabeled_only and labeled_only cannot both be True — pick one or set both to False to show all ROIs."

annotate_rois(data_path=ROI_DATA_PATH,
              n_samples=n_samples,
              unlabeled_only=unlabeled_only,
              labeled_only=labeled_only)

Found 14285 roi keys matching filter
Found 1000 unlabeled ROIs out of 14285 ROIs.
[1/1000] Updated: 2-4-1_139 → Good
[2/1000] Updated: 1-4-2_213 → Good
[3/1000] Updated: 2-4-3_324 → Good
[4/1000] Updated: 1-1-2_0001_369 → Good
[5/1000] Updated: 1-2-3_0 → Good
[6/1000] Updated: 1-2-1_302 → Good
[7/1000] Updated: 2-1_10uM_2m_385 → Good
[8/1000] Updated: 1-1-2_0001_206 → Bad
[9/1000] Updated: 2-3-1_91 → Good
[10/1000] Updated: 2-1-1_103 → Good
[11/1000] Updated: 2-1_551 → Good
[12/1000] Updated: 1-3-2_107 → Good
[13/1000] Updated: 2-4-1_248 → Good
[14/1000] Updated: 2-4-1_2 → Good
[15/1000] Updated: 1-1-3_88 → Good
[16/1000] Updated: 2-2_262 → Good
[17/1000] Updated: 2-3_301 → Bad
[18/1000] Updated: 1-1-1_17 → Bad
[19/1000] Updated: 1-4-3_389 → Good
[20/1000] Updated: 1-1-3_325 → Good
[21/1000] Updated: 2-3-3_323 → Good
[22/1000] Updated: 2-1_10uM_2m_86 → Bad
[23/1000] Updated: 2-1_10uM_2m_9 → Good
[24/1000] Updated: 2-1_719 → Good
[25/1000] Updated: 2-1_10uM_2m_538 → Good
[26/1000] Updat

{'level': 'roi',
 'queued': 1000,
 'total': 569,
 'labeled': 563,
 'updated': 559,
 'confirmed': 4,
 'skipped': 6}

In [7]:
name = "combined_roi_classifier" # TODO Change this as needed for your own experimental/organizational needs
data_paths = [ROI_DATA_PATH] # Can be a list of paths if you have engineered data from multiple sources you want to combine for training
results = train_roi_classifier(config_path=CONFIG_PATH, data_path=data_paths, name=name,
                     output_dir=MODEL_OUT_DIR, verbose=True, manual_only=True, overwrite=False)

Dataset Summary
--------------------------------------------------
Total labeled datapoints: 1047
  Train: 837 | Test: 210

Label distribution:
              Bad (0)  Good (1)
  Train           158       679
  Test             37       173
  Total           195       852

Training on: Manual labels only

--------------------------------------------------
TUNED MODEL SUMMARY
--------------------------------------------------
Model:     LogisticRegression
Transform: raw
Features:  ['derivative_skew', 'range_trace', 'ac_decay']

Hyperparameters:
  C: 100
  class_weight: None
  max_iter: 1000
  penalty: l1
  solver: saga

Metrics:
  CV Accuracy:   0.9701
  Test Accuracy: 0.9857
  ROC AUC:       0.9948
  F1:            0.9856
  Precision:     0.9856
  Recall:        0.9857

Confusion Matrix:
              Pred 0  Pred 1
  Actual 0    35      2      
  Actual 1    1       172    
--------------------------------------------------
Saved model to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\

c:\Users\mzinn1\AppData\Local\anaconda3\envs\gcamp\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
